# Commit watcher: gated commits with a hashed receipt (replayable)

`baebugs.watch_commit` (every 10 min on magicstudio, `mvp-kg` only): if allowed paths changed, regenerate the AoK and
the graph pack, run the AoK/MVP tests as a gate (fail closed), ask a local model for a one-line subject (Ollarma ->
Ollama -> deterministic), commit on the working branch (never `main`), push, and append a receipt to
`daisy/watcher_mmr.jsonl`. Model text varies; every step is hashed, so the variance is recorded.

This replay runs **one real cycle in a throw-away clone** with `push=False`, so the real repo is never touched.
**Swap data:** `SRC` (the repo to clone), `CHANGE` (the file the cycle should pick up), `RECEIPTS` (a real receipt
log to verify, e.g. the one on magicstudio).

In [1]:
import os, sys, json
from pathlib import Path
if Path.cwd().name == 'notebooks': os.chdir('..')
sys.path.insert(0, os.getcwd())
import subprocess, tempfile, hashlib
from baebugs import watch_commit as W
from baebugs.mmr import MMRLog

SRC = os.getcwd()                                   # swap: repo to clone
CHANGE = 'docs/watcher_replay_note.md'              # swap: a path under one of W.ALLOW
RECEIPTS = 'daisy/watcher_mmr.jsonl'                # swap: a real receipt log
print('allowed paths', W.ALLOW, '| model', W.MODEL)

allowed paths ['daisy/', 'kg/', 'site/mvp/', 'docs/', 'dashboard/csv/', 'dashboard/data/'] | model qwen2.5-coder:1.5b


## The subject line: local model if one answers, otherwise a deterministic fallback

In [2]:
stat = ' daisy/x.json | 2 +-\n kg/aok.json | 4 ++--\n'
subject, route, prompt = W.summarise(stat)
print(route, '|', subject, '| prompt sha', hashlib.sha256(prompt.encode()).hexdigest()[:12])

deterministic | update 2 file(s) under daisy, dashboard, docs, kg, site | prompt sha d619d1573720


## One cycle in a scratch clone

In [3]:
tmp = Path(tempfile.mkdtemp(prefix='watcher_replay_'))
g = lambda *a: subprocess.run(['git', '-C', str(tmp), *a], capture_output=True, text=True, check=True).stdout
subprocess.run(['git', 'clone', '-q', SRC, str(tmp)], check=True)
g('checkout', '-q', '-B', 'watcher-replay'); g('config', 'user.email', 'replay@example.invalid'); g('config', 'user.name', 'replay')
here = os.getcwd(); os.chdir(tmp)
try:
    print('no change ->', W.once(push=False))
    Path(CHANGE).parent.mkdir(parents=True, exist_ok=True); Path(CHANGE).write_text('replayed by notebooks/watch_commit.ipynb\n')
    rec = W.once(push=False)
    print(json.dumps({k: rec.get(k) for k in ('skipped', 'subject', 'route', 'commit', 'mvp_root')}, indent=1))
    g('checkout', '-q', '-B', 'main')
    print('on main ->', W.once(push=False))
finally:
    os.chdir(here)

no change -> {'skipped': 'nothing changed'}


{
 "skipped": null,
 "subject": "update 2 file(s) under daisy, dashboard, docs, kg, site",
 "route": "deterministic",
 "commit": "1722cb56ff9060ce755f0e4c64426c81d8dbd134",
 "mvp_root": "7e3150f9a44bf196ad96b5345c3130cea2921caa43869bf52eca6fec0bd271b1"
}
on main -> {'skipped': 'on main; the watcher never commits to main'}


## Check the receipt: MMR verifies, the commit exists, and its graph root is the pack inside that commit

In [4]:
if 'commit' in rec:
    log = MMRLog(tmp / 'daisy/watcher_mmr.jsonl')
    print('MMR verify', log.verify(), '| receipts', len(log.records))
    packed = json.loads(g('show', f"{rec['commit']}:site/mvp/mvp.json"))
    assert packed['root'] == rec['mvp_root'] and g('cat-file', '-t', rec['commit']).strip() == 'commit'
    assert hashlib.sha256(rec['subject'].encode()).hexdigest() == rec['response_sha256']
    print('receipt resolves: commit', rec['commit'][:10], '| mvp root', rec['mvp_root'][:12], '| subject hash ok')

MMR verify (True, None) | receipts 1
receipt resolves: commit 1722cb56ff | mvp root 7e3150f9a44b | subject hash ok


## Verify a real receipt log (if you have one here)

In [5]:
if Path(RECEIPTS).exists():
    log = MMRLog(RECEIPTS)
    ok, at = log.verify()
    miss = [r['record']['commit'] for r in log.records
            if subprocess.run(['git', 'cat-file', '-e', r['record']['commit']], capture_output=True).returncode]
    print('MMR', ok, at, '|', len(log.records), 'receipts |', len(miss), 'commits not in this clone (fetch first, else BREAK)')
else:
    print(RECEIPTS, 'not here: the live log is on magicstudio; swap RECEIPTS to it')

daisy/watcher_mmr.jsonl not here: the live log is on magicstudio; swap RECEIPTS to it
